<a href="https://colab.research.google.com/github/scorchbot/fantasy-football-picker/blob/main/Fantasy_Football_Picker_Clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fantasy Football Picker — Clean Pipeline

This notebook is the restart-safe version of the fantasy football research prototype.

**Normal startup:** Runtime → Run all.

The notebook rebuilds the historical dataset, trains the current position models, calibrates start/sit confidence with walk-forward predictions, and exposes `start_sit_2024()` for historical testing.

The next product milestone is to generalize scoring/predictions for arbitrary league settings and current-week use.


## 1. Setup


In [1]:
!pip install nflreadpy -q

import itertools
import re
import numpy as np
import pandas as pd
import nflreadpy as nfl

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

print("Setup complete")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
Setup complete


## 2. Load player-game data and apply current Yahoo scoring


In [2]:
PLAYER_STATS_URL = "https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats.csv"

players = pd.read_csv(PLAYER_STATS_URL)

recent = players[
    (players['season'] >= 2021) &
    (players['season'] <= 2024)
].copy()

print(f"Downloaded {len(players):,} player-game records")
print(f"Using {len(recent):,} records from 2021–2024")


Downloaded 134,470 player-game records
Using 22,579 records from 2021–2024


In [3]:
def calculate_fantasy_points(row):
    points = 0

    # Passing
    points += row['completions'] * -0.1
    points += row['passing_yards'] / 30
    points += row['passing_tds'] * 5
    points += row['interceptions'] * -2

    # Rushing
    points += row['rushing_yards'] / 10
    points += row['rushing_tds'] * 6

    # Rushing yardage bonuses
    if row['rushing_yards'] >= 300:
        points += 5
    elif row['rushing_yards'] >= 200:
        points += 3
    elif row['rushing_yards'] >= 150:
        points += 2

    # Receiving
    points += row['receptions'] * 0.5
    points += row['receiving_yards'] / 10
    points += row['receiving_tds'] * 6

    # Receiving yardage bonuses
    if row['receiving_yards'] >= 250:
        points += 3
    elif row['receiving_yards'] >= 200:
        points += 2
    elif row['receiving_yards'] >= 150:
        points += 1

    # 2-point conversions
    points += (
        row['passing_2pt_conversions']
        + row['rushing_2pt_conversions']
        + row['receiving_2pt_conversions']
    ) * 2

    # Fumbles lost
    points += (
        row['rushing_fumbles_lost']
        + row['receiving_fumbles_lost']
        + row['sack_fumbles_lost']
    ) * -2

    # Passing yardage bonuses
    if row['passing_yards'] >= 500:
        points += 4
    elif row['passing_yards'] >= 400:
        points += 3
    elif row['passing_yards'] >= 300:
        points += 2

    return round(points, 2)


In [4]:
recent['my_fantasy_points'] = recent.apply(
    calculate_fantasy_points,
    axis=1
)

print("Custom Yahoo scoring applied.")


Custom Yahoo scoring applied.


## 3. Schedule and game context


In [5]:
schedule_url = "https://github.com/nflverse/nflverse-data/releases/download/schedules/games.csv"

schedule = pd.read_csv(schedule_url)

print(f"Downloaded {len(schedule):,} games")
print(schedule.columns.tolist())


Downloaded 7,548 games
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [6]:
# Build one row per team per game

home_games = schedule[[
    'game_id',
    'season',
    'week',
    'gameday',
    'home_team',
    'away_team',
    'spread_line',
    'total_line',
    'roof',
    'temp',
    'wind',
    'home_rest',
    'home_qb_name'
]].copy()

home_games = home_games.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_rest': 'rest_days',
    'home_qb_name': 'starting_qb'
})

home_games['is_home'] = True
home_games['team_spread'] = -home_games['spread_line']


away_games = schedule[[
    'game_id',
    'season',
    'week',
    'gameday',
    'away_team',
    'home_team',
    'spread_line',
    'total_line',
    'roof',
    'temp',
    'wind',
    'away_rest',
    'away_qb_name'
]].copy()

away_games = away_games.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_rest': 'rest_days',
    'away_qb_name': 'starting_qb'
})

away_games['is_home'] = False
away_games['team_spread'] = away_games['spread_line']


team_games = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

print(f"Created {len(team_games):,} team-game records")


Created 15,096 team-game records


In [7]:
model_data = recent.merge(
    team_games,
    left_on=['season', 'week', 'recent_team'],
    right_on=['season', 'week', 'team'],
    how='left'
)

print(f"Player-game rows after schedule join: {len(model_data):,}")
print(f"Rows missing schedule data: {model_data['game_id'].isna().sum():,}")


Player-game rows after schedule join: 22,579
Rows missing schedule data: 0


## 4. Pregame player form and opponent-defense features


In [8]:
# Keep only regular-season games and fantasy-relevant offensive positions
model_data = model_data[
    (model_data['season_type'] == 'REG') &
    (model_data['position'].isin(['QB', 'RB', 'WR', 'TE']))
].copy()

# Put games in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# Rebuild our prior-3-game features,
# restarting the calculation each season
for col in [
    'my_fantasy_points',
    'carries',
    'targets',
    'receptions',
    'rushing_yards',
    'receiving_yards',
    'target_share'
]:
    model_data[f'{col}_last3'] = (
        model_data
        .groupby(['player_id', 'season'])[col]
        .transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    )

print(f"Fantasy-relevant regular-season player-games: {len(model_data):,}")


Fantasy-relevant regular-season player-games: 21,071


In [9]:
# Build a table of fantasy points allowed by each defense to each position
defense_games = (
    model_data
    .groupby([
        'season',
        'week',
        'opponent_team',
        'position'
    ])['my_fantasy_points']
    .sum()
    .reset_index()
    .rename(columns={
        'opponent_team': 'defense_team',
        'my_fantasy_points': 'fantasy_points_allowed'
    })
)

# Sort in time order
defense_games = defense_games.sort_values(
    ['defense_team', 'position', 'season', 'week']
).copy()

# Prior 3-game average fantasy points allowed
defense_games['defense_fp_allowed_last3'] = (
    defense_games
    .groupby(['defense_team', 'position', 'season'])['fantasy_points_allowed']
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

defense_games.head()


,season,week,defense_team,position,fantasy_points_allowed,defense_fp_allowed_last3
0,2021,1,ARI,QB,11.67,NaN
128,2021,2,ARI,QB,24.43,11.670000
256,2021,3,ARI,QB,4.80,18.050000
384,2021,4,ARI,QB,16.83,13.633333
512,2021,5,ARI,QB,11.80,15.353333


In [10]:
model_data = model_data.merge(
    defense_games[[
        'season',
        'week',
        'defense_team',
        'position',
        'defense_fp_allowed_last3'
    ]],
    left_on=[
        'season',
        'week',
        'opponent_team',
        'position'
    ],
    right_on=[
        'season',
        'week',
        'defense_team',
        'position'
    ],
    how='left'
)

print(
    "Rows with defensive matchup data:",
    model_data['defense_fp_allowed_last3'].notna().sum()
)

print(
    "Rows missing defensive matchup data:",
    model_data['defense_fp_allowed_last3'].isna().sum()
)


Rows with defensive matchup data: 19844
Rows missing defensive matchup data: 1227


## 5. Snap-count features


In [11]:
snap_counts = nfl.load_snap_counts([2021, 2022, 2023, 2024])
snap_pd = snap_counts.to_pandas()
snap_pd = snap_pd[
    snap_pd['position'].isin(['QB', 'RB', 'WR', 'TE'])
].copy()

print(f"Loaded {len(snap_pd):,} fantasy-relevant snap-count rows")


Loaded 28,743 fantasy-relevant snap-count rows


In [12]:
def normalize_name(name):
    if pd.isna(name):
        return name

    name = name.lower()
    name = re.sub(r"[.'’\-]", "", name)
    name = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

aliases = {
    'gabe davis': 'gabriel davis',
    'jeffery wilson': 'jeff wilson',
    'chig okonkwo': 'chigoziem okonkwo',
    'joseph fortson': 'jody fortson',
    'mike woods': 'michael woods',
    'christopher brooks': 'chris brooks',
    'christopher herndon': 'chris herndon',
    'dee eskridge': 'dwayne eskridge',
    'bam knight': 'zonovan knight',
}

model_data['name_norm'] = (
    model_data['player_display_name']
    .apply(normalize_name)
    .replace(aliases)
)
snap_pd['name_norm'] = snap_pd['player'].apply(normalize_name)

# Make this merge safe to rerun.
snap_helper_cols = [
    'player', 'team', 'opponent', 'offense_snaps', 'offense_pct'
]
model_data = model_data.drop(
    columns=[c for c in snap_helper_cols if c in model_data.columns]
)

model_data = model_data.merge(
    snap_pd[[
        'season', 'week', 'name_norm', 'team', 'opponent',
        'offense_snaps', 'offense_pct'
    ]],
    left_on=[
        'season', 'week', 'name_norm', 'recent_team', 'opponent_team'
    ],
    right_on=[
        'season', 'week', 'name_norm', 'team', 'opponent'
    ],
    how='left'
)

print("Snap rows matched:", model_data['offense_pct'].notna().sum())
print("Snap rows missing:", model_data['offense_pct'].isna().sum())
print(f"Snap match rate: {model_data['offense_pct'].notna().mean() * 100:.2f}%")


Snap rows matched: 21058
Snap rows missing: 13
Snap match rate: 99.94%


In [13]:
# Make sure rows are in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# Prior 3-game and 5-game snap share
model_data['offense_pct_last3'] = (
    model_data
    .groupby(['player_id', 'season'])['offense_pct']
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

model_data['offense_pct_last5'] = (
    model_data
    .groupby(['player_id', 'season'])['offense_pct']
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

model_data['snap_trend'] = (
    model_data['offense_pct_last3'] -
    model_data['offense_pct_last5']
)

print("Snap-share features created.")


Snap-share features created.


## 6. QB-specific pregame features


In [14]:
qb_cols = [
    'completions',
    'attempts',
    'passing_yards',
    'passing_tds',
    'interceptions',
    'passing_air_yards',
    'passing_epa',
    'carries',
    'rushing_yards',
    'rushing_tds',
    'my_fantasy_points'
]

for col in qb_cols:
    model_data[f'{col}_last3'] = (
        model_data
        .groupby(['player_id', 'season'])[col]
        .transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    )

print("QB pregame features created.")


QB pregame features created.


## 7. Injury features


In [15]:
injuries = nfl.load_injuries([2021, 2022, 2023, 2024])
injury_pd = injuries.to_pandas()
print(f"Loaded {len(injury_pd):,} injury-report rows")


Loaded 23,083 injury-report rows


In [16]:
# Keep the fields we need
injury_features = injury_pd[[
    'season',
    'week',
    'team',
    'gsis_id',
    'report_status',
    'practice_status'
]].copy()

# Convert statuses into model-friendly flags
injury_features['on_injury_report'] = 1

injury_features['questionable'] = (
    injury_features['report_status'] == 'Questionable'
).astype(int)

injury_features['doubtful'] = (
    injury_features['report_status'] == 'Doubtful'
).astype(int)

injury_features['out'] = (
    injury_features['report_status'] == 'Out'
).astype(int)

injury_features['practice_full'] = (
    injury_features['practice_status'] == 'Full Participation in Practice'
).astype(int)

injury_features['practice_limited'] = (
    injury_features['practice_status'] == 'Limited Participation in Practice'
).astype(int)

injury_features['practice_dnp'] = (
    injury_features['practice_status'] == 'Did Not Participate In Practice'
).astype(int)

# Keep one record per player/week/team
injury_features = injury_features.drop_duplicates(
    subset=['season', 'week', 'team', 'gsis_id']
)

injury_features.head()


,season,week,team,gsis_id,report_status,practice_status,on_injury_report,questionable,doubtful,out,practice_full,practice_limited,practice_dnp
0,2021,1,ARI,00-0033258,None,Full Participation in Practice,1,0,0,0,1,0,0
1,2021,1,ARI,00-0034473,Out,Did Not Participate In Practice,1,0,0,1,0,0,1
2,2021,1,ARI,00-0035126,Out,Did Not Participate In Practice,1,0,0,1,0,0,1
3,2021,1,ATL,00-0031583,None,Did Not Participate In Practice,1,0,0,0,0,0,1
4,2021,1,ATL,00-0030010,None,Full Participation in Practice,1,0,0,0,1,0,0


In [17]:
# Injury feature column names
injury_cols = [
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp'
]

# Remove any leftovers from prior runs/merges
columns_to_remove = [
    'team',
    'team_x',
    'team_y',
    'opponent',
    'opponent_x',
    'opponent_y',
    'gsis_id',
    'injury_team',
    'injury_player_id',

    # Old injury columns
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp',

    # Possible suffixed versions from failed/repeated merges
    'on_injury_report_x',
    'on_injury_report_y',
    'questionable_x',
    'questionable_y',
    'doubtful_x',
    'doubtful_y',
    'out_x',
    'out_y',
    'practice_full_x',
    'practice_full_y',
    'practice_limited_x',
    'practice_limited_y',
    'practice_dnp_x',
    'practice_dnp_y'
]

model_data = model_data.drop(
    columns=[c for c in columns_to_remove if c in model_data.columns]
)

# Prepare injury table
injury_merge = injury_features[[
    'season',
    'week',
    'team',
    'gsis_id',
    'on_injury_report',
    'questionable',
    'doubtful',
    'out',
    'practice_full',
    'practice_limited',
    'practice_dnp'
]].copy()

injury_merge = injury_merge.rename(columns={
    'team': 'injury_team',
    'gsis_id': 'injury_player_id'
})

# Merge cleanly
model_data = model_data.merge(
    injury_merge,
    left_on=[
        'season',
        'week',
        'recent_team',
        'player_id'
    ],
    right_on=[
        'season',
        'week',
        'injury_team',
        'injury_player_id'
    ],
    how='left'
)

# Players not listed that week get zeros
model_data[injury_cols] = model_data[injury_cols].fillna(0)

print("Injury features attached.")
print(model_data[injury_cols].sum())


Injury features attached.
on_injury_report    3781.0
questionable         988.0
doubtful               1.0
out                    0.0
practice_full       2325.0
practice_limited    1101.0
practice_dnp         336.0
dtype: float64


## 8. Expected-opportunity features


In [18]:
ff_opportunity = nfl.load_ff_opportunity([2021, 2022, 2023, 2024])
ff_opp_pd = ff_opportunity.to_pandas()
ff_opp_pd['season'] = ff_opp_pd['season'].astype(int)
ff_opp_pd['week'] = ff_opp_pd['week'].astype(int)

ff_opp_pd['my_expected_opportunity_points'] = (
    (ff_opp_pd['pass_completions_exp'] * -0.1) +
    (ff_opp_pd['pass_yards_gained_exp'] / 30) +
    (ff_opp_pd['pass_touchdown_exp'] * 5) +
    (ff_opp_pd['pass_interception_exp'] * -2) +
    (ff_opp_pd['rush_yards_gained_exp'] / 10) +
    (ff_opp_pd['rush_touchdown_exp'] * 6) +
    (ff_opp_pd['receptions_exp'] * 0.5) +
    (ff_opp_pd['rec_yards_gained_exp'] / 10) +
    (ff_opp_pd['rec_touchdown_exp'] * 6) +
    (
        ff_opp_pd['pass_two_point_conv_exp'] +
        ff_opp_pd['rush_two_point_conv_exp'] +
        ff_opp_pd['rec_two_point_conv_exp']
    ) * 2
)

print(f"Loaded {len(ff_opp_pd):,} opportunity rows")


Loaded 24,189 opportunity rows


In [19]:
# Make merge key types consistent
model_data['season'] = model_data['season'].astype(int)
model_data['week'] = model_data['week'].astype(int)

ff_opp_pd['season'] = ff_opp_pd['season'].astype(int)
ff_opp_pd['week'] = ff_opp_pd['week'].astype(int)


# Remove opportunity columns if this cell has been run before
opp_cols_to_remove = [
    'opp_team',
    'opp_player_id',
    'my_expected_opportunity_points',
    'rush_yards_gained_exp',
    'rec_yards_gained_exp',
    'receptions_exp',
    'rush_touchdown_exp',
    'rec_touchdown_exp',
    'pass_yards_gained_exp',
    'pass_touchdown_exp'
]

model_data = model_data.drop(
    columns=[c for c in opp_cols_to_remove if c in model_data.columns]
)


# Prepare opportunity data
opp_merge = ff_opp_pd[[
    'season',
    'week',
    'posteam',
    'player_id',
    'my_expected_opportunity_points',
    'rush_yards_gained_exp',
    'rec_yards_gained_exp',
    'receptions_exp',
    'rush_touchdown_exp',
    'rec_touchdown_exp',
    'pass_yards_gained_exp',
    'pass_touchdown_exp'
]].copy()

opp_merge = opp_merge.rename(columns={
    'posteam': 'opp_team',
    'player_id': 'opp_player_id'
})


# Merge into master dataset
model_data = model_data.merge(
    opp_merge,
    left_on=[
        'season',
        'week',
        'recent_team',
        'player_id'
    ],
    right_on=[
        'season',
        'week',
        'opp_team',
        'opp_player_id'
    ],
    how='left'
)

print("Rows:", len(model_data))
print(
    "Rows with opportunity data:",
    model_data['my_expected_opportunity_points'].notna().sum()
)
print(
    "Rows missing opportunity data:",
    model_data['my_expected_opportunity_points'].isna().sum()
)


Rows: 21071
Rows with opportunity data: 21049
Rows missing opportunity data: 22


In [20]:
# Put everything in chronological order
model_data = model_data.sort_values(
    ['player_id', 'season', 'week']
).copy()

# Opportunity metrics we want to know entering each game
opp_cols = [
    'my_expected_opportunity_points',
    'rush_yards_gained_exp',
    'rec_yards_gained_exp',
    'receptions_exp',
    'rush_touchdown_exp',
    'rec_touchdown_exp',
    'pass_yards_gained_exp',
    'pass_touchdown_exp'
]

# Create PRIOR 3-game averages
for col in opp_cols:
    model_data[f'{col}_last3'] = (
        model_data
        .groupby(['player_id', 'season'])[col]
        .transform(
            lambda x: x.shift(1).rolling(
                3,
                min_periods=1
            ).mean()
        )
    )

print("Pregame expected-opportunity features created.")


Pregame expected-opportunity features created.


## 9. Final feature sets


In [21]:
# Opportunity features that apply differently by position

opp_features = {

    'QB': [
        'my_expected_opportunity_points_last3',
        'pass_yards_gained_exp_last3',
        'pass_touchdown_exp_last3'
    ],

    'RB': [
        'my_expected_opportunity_points_last3',
        'rush_yards_gained_exp_last3',
        'rec_yards_gained_exp_last3',
        'receptions_exp_last3',
        'rush_touchdown_exp_last3',
        'rec_touchdown_exp_last3'
    ],

    'WR': [
        'my_expected_opportunity_points_last3',
        'rec_yards_gained_exp_last3',
        'receptions_exp_last3',
        'rec_touchdown_exp_last3'
    ],

    'TE': [
        'my_expected_opportunity_points_last3',
        'rec_yards_gained_exp_last3',
        'receptions_exp_last3',
        'rec_touchdown_exp_last3'
    ]
}


# Our current best feature sets

current_features = {

    'QB': [
        'my_fantasy_points_last3',
        'attempts_last3',
        'completions_last3',
        'passing_yards_last3',
        'passing_tds_last3',
        'interceptions_last3',
        'passing_air_yards_last3',
        'passing_epa_last3',
        'carries_last3',
        'rushing_yards_last3',
        'rushing_tds_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'is_home',
        'temp',
        'wind',
        'rest_days'
    ],

    'RB': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend',
        'on_injury_report',
        'questionable',
        'practice_full',
        'practice_limited',
        'practice_dnp'
    ],

    'WR': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend',
        'on_injury_report',
        'questionable',
        'practice_full',
        'practice_limited',
        'practice_dnp'
    ],

    'TE': [
        'my_fantasy_points_last3',
        'carries_last3',
        'targets_last3',
        'rushing_yards_last3',
        'receiving_yards_last3',
        'target_share_last3',
        'defense_fp_allowed_last3',
        'team_spread',
        'total_line',
        'offense_pct_last3',
        'offense_pct_last5',
        'snap_trend',
        'on_injury_report',
        'questionable',
        'practice_full',
        'practice_limited',
        'practice_dnp'
    ]
}


## 10. Train final historical position models


In [22]:
# Train and store the final position models we want to use

final_models = {}
final_features = {}
final_medians = {}

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[pos_data['season'] <= 2023].copy()

    # Expected opportunity helped QB ranking slightly,
    # so keep it for QB only.
    if pos == 'QB':
        cols = current_features[pos] + opp_features[pos]
    else:
        cols = current_features[pos]

    X_train = train[cols].copy()

    medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(medians)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        train['my_fantasy_points']
    )

    final_models[pos] = model
    final_features[pos] = cols
    final_medians[pos] = medians

print("Final QB/RB/WR/TE models trained and stored.")


Final QB/RB/WR/TE models trained and stored.


## 11. Walk-forward confidence calibration


In [23]:
walkforward_predictions = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == pos) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    cols = final_features[pos]

    # Predict each season using ONLY previous seasons
    for test_year in [2022, 2023, 2024]:

        train = pos_data[
            pos_data['season'] < test_year
        ].copy()

        test = pos_data[
            pos_data['season'] == test_year
        ].copy()

        if len(train) == 0 or len(test) == 0:
            continue

        X_train = train[cols].copy()
        X_test = test[cols].copy()

        medians = X_train.median(numeric_only=True)

        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            train['my_fantasy_points']
        )

        test['prediction'] = model.predict(X_test)
        test['Position'] = pos

        walkforward_predictions.append(
            test[[
                'season',
                'week',
                'player_display_name',
                'recent_team',
                'opponent_team',
                'Position',
                'prediction',
                'my_fantasy_points'
            ]]
        )

walkforward_df = pd.concat(
    walkforward_predictions,
    ignore_index=True
)

print("Walk-forward predictions created.")
print("Rows:", len(walkforward_df))

print(
    walkforward_df
    .groupby(['Position', 'season'])
    .size()
)


Walk-forward predictions created.
Rows: 14094
Position  season
QB        2022       564
          2023       582
          2024       586
RB        2022      1236
          2023      1209
          2024      1208
TE        2022       965
          2023       959
          2024       969
WR        2022      1912
          2023      1999
          2024      1905
dtype: int64


In [24]:
walkforward_decisions = []

realistic_pool_sizes = {
    'QB': 18,
    'RB': 30,
    'WR': 40,
    'TE': 18
}

for pos in ['QB', 'RB', 'WR', 'TE']:

    pos_data = walkforward_df[
        walkforward_df['Position'] == pos
    ].copy()

    for (season, week), group in pos_data.groupby(['season', 'week']):

        # Limit to realistic fantasy-relevant pool
        group = group.nlargest(
            realistic_pool_sizes[pos],
            'prediction'
        )

        rows = group.to_dict('records')

        # Every possible 3-player decision
        for trio in itertools.combinations(rows, 3):

            trio = sorted(
                trio,
                key=lambda x: x['prediction'],
                reverse=True
            )

            model_pick = trio[0]

            gap = (
                trio[0]['prediction'] -
                trio[1]['prediction']
            )

            actual_scores = [
                p['my_fantasy_points']
                for p in trio
            ]

            max_actual = max(actual_scores)

            winners = [
                p for p in trio
                if p['my_fantasy_points'] == max_actual
            ]

            # Ignore exact ties for highest scorer
            if len(winners) != 1:
                continue

            correct = (
                model_pick['player_display_name']
                ==
                winners[0]['player_display_name']
            )

            walkforward_decisions.append({
                'Position': pos,
                'season': season,
                'week': week,
                'gap': gap,
                'correct': correct
            })

walkforward_decisions = pd.DataFrame(
    walkforward_decisions
)

print("Walk-forward 3-player decisions created.")
print("Total decisions:", len(walkforward_decisions))

walkforward_decisions.groupby('Position').agg(
    decisions=('correct', 'size'),
    overall_accuracy=('correct', 'mean')
)


Walk-forward 3-player decisions created.
Total decisions: 790522


,decisions,overall_accuracy
Position,,
QB,41566,0.433672
RB,206015,0.424474
TE,41379,0.433432
WR,501562,0.429624


In [25]:
from sklearn.linear_model import LogisticRegression
import numpy as np

confidence_models = {}
confidence_summary = []

for pos in ['QB', 'RB', 'WR', 'TE']:

    df = walkforward_decisions[
        walkforward_decisions['Position'] == pos
    ].copy()

    X = df[['gap']].values
    y = df['correct'].astype(int).values

    clf = LogisticRegression()
    clf.fit(X, y)

    confidence_models[pos] = clf

    # Show estimated confidence at useful projection gaps
    for gap in [0, 1, 2, 3, 4, 5, 7.5, 10]:

        prob = clf.predict_proba(
            np.array([[gap]])
        )[0, 1]

        confidence_summary.append({
            'Position': pos,
            'Projection gap': gap,
            'Estimated confidence': round(prob, 3)
        })

confidence_curve_table = pd.DataFrame(
    confidence_summary
)

confidence_curve_table


,Position,Projection gap,Estimated confidence
0,QB,0.0,0.378
1,QB,1.0,0.401
2,QB,2.0,0.425
3,QB,3.0,0.449
4,QB,4.0,0.473
5,QB,5.0,0.498
6,QB,7.5,0.559
7,QB,10.0,0.618
8,RB,0.0,0.375
9,RB,1.0,0.395


## 12. Historical start/sit engine


In [26]:
def get_confidence(position, gap):

    position = position.upper()

    if position not in confidence_models:
        return None

    probability = confidence_models[position].predict_proba(
        np.array([[gap]])
    )[0, 1]

    return probability


def confidence_label(confidence):

    if confidence is None:
        return "Unknown"

    if confidence < 0.45:
        return "Very Low"
    elif confidence < 0.50:
        return "Low"
    elif confidence < 0.60:
        return "Moderate"
    elif confidence < 0.70:
        return "High"
    else:
        return "Very High"


def explain_model_drivers(selected, position):

    top = selected.iloc[0].copy()

    cols = final_features[position]
    model = final_models[position]
    medians = final_medians[position]

    # Create the top player's original feature row
    original = pd.DataFrame(
        [top[cols]]
    ).copy()

    original = original.fillna(medians)

    original_prediction = model.predict(
        original
    )[0]

    driver_rows = []

    friendly_names = {
        'my_fantasy_points_last3':
            'Recent fantasy production',

        'carries_last3':
            'Recent carries',

        'targets_last3':
            'Recent targets',

        'rushing_yards_last3':
            'Recent rushing yards',

        'receiving_yards_last3':
            'Recent receiving yards',

        'target_share_last3':
            'Recent target share',

        'offense_pct_last3':
            'Recent snap share',

        'offense_pct_last5':
            'Longer-term snap share',

        'snap_trend':
            'Recent snap trend',

        'team_spread':
            'Game spread',

        'total_line':
            'Game total',

        'defense_fp_allowed_last3':
            'Opponent matchup',

        'passing_yards_last3':
            'Recent passing yards',

        'passing_tds_last3':
            'Recent passing TDs',

        'attempts_last3':
            'Recent pass attempts',

        'completions_last3':
            'Recent completions',

        'passing_air_yards_last3':
            'Recent passing air yards',

        'passing_epa_last3':
            'Recent passing efficiency',

        'rushing_tds_last3':
            'Recent rushing TDs',

        'interceptions_last3':
            'Recent interceptions',

        'is_home':
            'Home-field status',

        'temp':
            'Temperature',

        'wind':
            'Wind',

        'rest_days':
            'Rest advantage',

        'on_injury_report':
            'Injury-report status',

        'questionable':
            'Questionable status',

        'practice_full':
            'Full-practice status',

        'practice_limited':
            'Limited-practice status',

        'practice_dnp':
            'Missed-practice status',

        'my_expected_opportunity_points_last3':
            'Recent expected opportunity',

        'pass_yards_gained_exp_last3':
            'Recent expected passing yards',

        'pass_touchdown_exp_last3':
            'Recent expected passing TDs'
    }

    # Test each feature by replacing it with
    # the training-set median and seeing how
    # much the projection changes
    for col in cols:

        if col not in medians.index:
            continue

        altered = original.copy()

        altered.loc[
            altered.index[0],
            col
        ] = medians[col]

        altered_prediction = model.predict(
            altered
        )[0]

        impact = (
            original_prediction
            - altered_prediction
        )

        driver_rows.append({
            'feature': col,
            'label': friendly_names.get(
                col,
                col
            ),
            'impact': impact
        })

    drivers = pd.DataFrame(
        driver_rows
    )

    if len(drivers) == 0:
        return drivers

    drivers = drivers.sort_values(
        'impact',
        ascending=False
    )

    # We only want features currently helping
    # the recommended player's projection
    drivers = drivers[
        drivers['impact'] > 0
    ]

    return drivers.head(5)


def build_caution_reasons(
    selected,
    position,
    gap,
    confidence
):

    cautions = []

    if confidence is None:
        return cautions

    # General confidence warning
    if confidence < 0.50:

        cautions.append(
            "This is a low-confidence decision based on "
            "historical 3-player start/sit results."
        )

    # Projection-gap warning
    if gap is not None and gap < 2:

        cautions.append(
            "The top two players are separated by less "
            "than 2 projected points."
        )

    elif gap is not None and gap < 5:

        cautions.append(
            "The projection gap is meaningful, but not "
            "large enough to make this a strong call."
        )

    # Compare top two players
    if len(selected) >= 2:

        top = selected.iloc[0]
        second = selected.iloc[1]

        # Similar recent fantasy production
        if (
            'my_fantasy_points_last3'
            in selected.columns
            and pd.notna(
                top['my_fantasy_points_last3']
            )
            and pd.notna(
                second['my_fantasy_points_last3']
            )
        ):

            recent_gap = abs(
                top['my_fantasy_points_last3']
                - second['my_fantasy_points_last3']
            )

            if recent_gap < 2:

                cautions.append(
                    f"{second['player_display_name']} has "
                    "very similar recent fantasy production."
                )

        # Position-specific workload comparison
        workload_col = None

        if position == 'RB':
            workload_col = 'carries_last3'

        elif position in ['WR', 'TE']:
            workload_col = 'targets_last3'

        elif position == 'QB':
            workload_col = 'attempts_last3'

        if (
            workload_col is not None
            and workload_col in selected.columns
            and pd.notna(
                top[workload_col]
            )
            and pd.notna(
                second[workload_col]
            )
        ):

            workload_gap = abs(
                top[workload_col]
                - second[workload_col]
            )

            if workload_gap < 2:

                cautions.append(
                    f"{second['player_display_name']} has "
                    "a very similar recent workload."
                )

    return cautions[:3]


def start_sit_2024(
    week,
    position,
    players
):

    position = position.upper()

    if position not in [
        'QB',
        'RB',
        'WR',
        'TE'
    ]:

        print(
            "Position must be QB, RB, WR, or TE."
        )

        return

    # -----------------------------------
    # GET PLAYERS FOR THIS WEEK
    # -----------------------------------

    week_data = model_data[
        (model_data['season'] == 2024) &
        (model_data['week'] == week) &
        (model_data['position'] == position)
    ].copy()

    selected = week_data[
        week_data[
            'player_display_name'
        ].isin(players)
    ].copy()

    if len(selected) == 0:

        print(
            "No matching players found."
        )

        return

    # Warn if a supplied player wasn't found
    found = set(
        selected[
            'player_display_name'
        ]
    )

    missing = [
        p
        for p in players
        if p not in found
    ]

    if missing:

        print(
            "Could not find:",
            missing
        )

    # -----------------------------------
    # CREATE MODEL PROJECTIONS
    # -----------------------------------

    cols = final_features[
        position
    ]

    X = selected[
        cols
    ].copy()

    X = X.fillna(
        final_medians[
            position
        ]
    )

    selected[
        'model_projection'
    ] = (
        final_models[
            position
        ].predict(X)
    )

    selected = (
        selected
        .sort_values(
            'model_projection',
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    selected[
        'rank'
    ] = (
        selected.index + 1
    )

    # -----------------------------------
    # CONFIDENCE
    # -----------------------------------

    if len(selected) >= 2:

        gap = (
            selected.loc[
                0,
                'model_projection'
            ]
            - selected.loc[
                1,
                'model_projection'
            ]
        )

        confidence = (
            get_confidence(
                position,
                gap
            )
        )

    else:

        gap = None
        confidence = None

    label = confidence_label(
        confidence
    )

    # -----------------------------------
    # MODEL-AWARE EXPLANATION
    # -----------------------------------

    driver_table = (
        explain_model_drivers(
            selected,
            position
        )
    )

    reasons = (
        driver_table.head(3)
    )

    cautions = (
        build_caution_reasons(
            selected,
            position,
            gap,
            confidence
        )
    )

    # -----------------------------------
    # HISTORICAL RESULT
    # -----------------------------------

    actual_winner_idx = (
        selected[
            'my_fantasy_points'
        ].idxmax()
    )

    actual_winner = (
        selected.loc[
            actual_winner_idx,
            'player_display_name'
        ]
    )

    recommendation = (
        selected.loc[
            0,
            'player_display_name'
        ]
    )

    correct_pick = (
        recommendation
        == actual_winner
    )

    # -----------------------------------
    # DISPLAY
    # -----------------------------------

    print()
    print(
        "=" * 50
    )

    print(
        f"2024 WEEK {week} — "
        f"{position} START/SIT"
    )

    print(
        "=" * 50
    )

    print()
    print(
        f"START: {recommendation}"
    )

    print()
    print(
        "MODEL RANKINGS"
    )

    for i, row in selected.iterrows():

        print(
            f"{i + 1}. "
            f"{row['player_display_name']} "
            f"({row['recent_team']} vs "
            f"{row['opponent_team']}) "
            f"— "
            f"{row['model_projection']:.2f} pts"
        )

    # -----------------------------------
    # CONFIDENCE DISPLAY
    # -----------------------------------

    if confidence is not None:

        print()

        print(
            f"Projection gap over #2: "
            f"{gap:.2f} points"
        )

        print(
            f"Historical confidence: "
            f"{confidence:.1%} "
            f"({label})"
        )

    # -----------------------------------
    # WHY THE MODEL LIKES THE PICK
    # -----------------------------------

    print()
    print(
        "WHY THE MODEL LIKES THIS PICK"
    )

    if len(reasons) > 0:

        for _, driver in reasons.iterrows():

            print(
                f"• {driver['label']} "
                f"is a strong positive driver "
                f"of the model's projection"
            )

    else:

        print(
            "• No single feature clearly "
            "drives the recommendation."
        )

    # -----------------------------------
    # REASONS FOR CAUTION
    # -----------------------------------

    if cautions:

        print()
        print(
            "REASONS FOR CAUTION"
        )

        for caution in cautions:

            print(
                f"• {caution}"
            )

    # -----------------------------------
    # HISTORICAL RESULT DISPLAY
    # -----------------------------------

    print()
    print(
        "HISTORICAL RESULT"
    )

    print(
        f"Actual top scorer: "
        f"{actual_winner}"
    )

    if correct_pick:

        print(
            "Model result: CORRECT"
        )

    else:

        print(
            "Model result: MISSED"
        )

    print()

    # -----------------------------------
    # RETURN TABLE
    # -----------------------------------

    result = selected[[
        'rank',
        'player_display_name',
        'recent_team',
        'opponent_team',
        'model_projection',
        'my_fantasy_points'
    ]].copy()

    result[
        'model_projection'
    ] = (
        result[
            'model_projection'
        ].round(2)
    )

    result[
        'my_fantasy_points'
    ] = (
        result[
            'my_fantasy_points'
        ].round(2)
    )

    return result


### Example historical decision


In [27]:
start_sit_2024(
    week=10,
    position='RB',
    players=[
        'Saquon Barkley',
        'Bijan Robinson',
        'Breece Hall'
    ]
)



2024 WEEK 10 — RB START/SIT

START: Saquon Barkley

MODEL RANKINGS
1. Saquon Barkley (PHI vs DAL) — 20.45 pts
2. Bijan Robinson (ATL vs NO) — 16.61 pts
3. Breece Hall (NYJ vs ARI) — 13.81 pts

Projection gap over #2: 3.84 points
Historical confidence: 45.6% (Low)

WHY THE MODEL LIKES THIS PICK
• Recent snap share is a strong positive driver of the model's projection
• Recent carries is a strong positive driver of the model's projection
• Recent fantasy production is a strong positive driver of the model's projection

REASONS FOR CAUTION
• This is a low-confidence decision based on historical 3-player start/sit results.
• The projection gap is meaningful, but not large enough to make this a strong call.

HISTORICAL RESULT
Actual top scorer: Bijan Robinson
Model result: MISSED



,rank,player_display_name,recent_team,opponent_team,model_projection,my_fantasy_points
0,1,Saquon Barkley,PHI,DAL,20.45,8.3
1,2,Bijan Robinson,ATL,NO,16.61,27.9
2,3,Breece Hall,NYJ,ARI,13.81,10.3


## 13. Checkpoint / next milestone


In [28]:
print("=== PROJECT CHECKPOINT ===")
print("Clean historical pipeline loaded successfully.")
print("Historical start/sit engine is ready.")
print("Next milestone: multi-league scoring architecture and stat-component prediction models.")


=== PROJECT CHECKPOINT ===
Clean historical pipeline loaded successfully.
Historical start/sit engine is ready.
Next milestone: multi-league scoring architecture and stat-component prediction models.


In [29]:
# ============================================================
# MULTI-LEAGUE SCORING ENGINE
# ============================================================

DEFAULT_LEAGUE_SETTINGS = {
    # Passing
    'completion_points': -0.1,
    'passing_yards_per_point': 30,
    'passing_td_points': 5,
    'interception_points': -2,

    # Passing bonuses
    'passing_bonus_300': 2,
    'passing_bonus_400': 3,
    'passing_bonus_500': 4,

    # Rushing
    'rushing_yards_per_point': 10,
    'rushing_td_points': 6,

    # Rushing bonuses
    'rushing_bonus_150': 2,
    'rushing_bonus_200': 3,
    'rushing_bonus_300': 5,

    # Receiving
    'reception_points': 0.5,
    'receiving_yards_per_point': 10,
    'receiving_td_points': 6,

    # Receiving bonuses
    'receiving_bonus_150': 1,
    'receiving_bonus_200': 2,
    'receiving_bonus_250': 3,

    # Misc.
    'two_point_conversion_points': 2,
    'fumble_lost_points': -2
}


def get_stat(stats, stat_name):
    """
    Safely get a stat from either a dictionary
    or a pandas Series.
    Missing stats are treated as zero.
    """

    value = stats.get(stat_name, 0)

    if pd.isna(value):
        return 0

    return value


def score_player(stats, settings):

    points = 0.0

    # --------------------------------------------------------
    # PASSING
    # --------------------------------------------------------

    completions = get_stat(
        stats,
        'completions'
    )

    passing_yards = get_stat(
        stats,
        'passing_yards'
    )

    passing_tds = get_stat(
        stats,
        'passing_tds'
    )

    interceptions = get_stat(
        stats,
        'interceptions'
    )

    points += (
        completions
        * settings['completion_points']
    )

    points += (
        passing_yards
        / settings['passing_yards_per_point']
    )

    points += (
        passing_tds
        * settings['passing_td_points']
    )

    points += (
        interceptions
        * settings['interception_points']
    )

    # Passing yard bonuses
    if passing_yards >= 500:

        points += settings[
            'passing_bonus_500'
        ]

    elif passing_yards >= 400:

        points += settings[
            'passing_bonus_400'
        ]

    elif passing_yards >= 300:

        points += settings[
            'passing_bonus_300'
        ]

    # --------------------------------------------------------
    # RUSHING
    # --------------------------------------------------------

    rushing_yards = get_stat(
        stats,
        'rushing_yards'
    )

    rushing_tds = get_stat(
        stats,
        'rushing_tds'
    )

    points += (
        rushing_yards
        / settings['rushing_yards_per_point']
    )

    points += (
        rushing_tds
        * settings['rushing_td_points']
    )

    # Rushing yard bonuses
    if rushing_yards >= 300:

        points += settings[
            'rushing_bonus_300'
        ]

    elif rushing_yards >= 200:

        points += settings[
            'rushing_bonus_200'
        ]

    elif rushing_yards >= 150:

        points += settings[
            'rushing_bonus_150'
        ]

    # --------------------------------------------------------
    # RECEIVING
    # --------------------------------------------------------

    receptions = get_stat(
        stats,
        'receptions'
    )

    receiving_yards = get_stat(
        stats,
        'receiving_yards'
    )

    receiving_tds = get_stat(
        stats,
        'receiving_tds'
    )

    points += (
        receptions
        * settings['reception_points']
    )

    points += (
        receiving_yards
        / settings['receiving_yards_per_point']
    )

    points += (
        receiving_tds
        * settings['receiving_td_points']
    )

    # Receiving yard bonuses
    if receiving_yards >= 250:

        points += settings[
            'receiving_bonus_250'
        ]

    elif receiving_yards >= 200:

        points += settings[
            'receiving_bonus_200'
        ]

    elif receiving_yards >= 150:

        points += settings[
            'receiving_bonus_150'
        ]

    # --------------------------------------------------------
    # TWO-POINT CONVERSIONS
    # --------------------------------------------------------

    two_point_conversions = (
        get_stat(
            stats,
            'passing_2pt_conversions'
        )
        + get_stat(
            stats,
            'rushing_2pt_conversions'
        )
        + get_stat(
            stats,
            'receiving_2pt_conversions'
        )
    )

    points += (
        two_point_conversions
        * settings[
            'two_point_conversion_points'
        ]
    )

    # --------------------------------------------------------
    # FUMBLES LOST
    # --------------------------------------------------------

    fumbles_lost = (
        get_stat(
            stats,
            'rushing_fumbles_lost'
        )
        + get_stat(
            stats,
            'receiving_fumbles_lost'
        )
        + get_stat(
            stats,
            'sack_fumbles_lost'
        )
    )

    points += (
        fumbles_lost
        * settings[
            'fumble_lost_points'
        ]
    )

    return round(points, 2)


print("Multi-league scoring engine created.")

Multi-league scoring engine created.


In [30]:
# Verify that the new generic scoring engine
# reproduces our existing Yahoo scoring

test_sample = recent.head(1000).copy()

test_sample['generic_score'] = test_sample.apply(
    lambda row: score_player(
        row,
        DEFAULT_LEAGUE_SETTINGS
    ),
    axis=1
)

test_sample['score_difference'] = (
    test_sample['generic_score']
    - test_sample['my_fantasy_points']
)

print(
    "Maximum scoring difference:",
    test_sample['score_difference'].abs().max()
)

print(
    "Average scoring difference:",
    test_sample['score_difference'].abs().mean()
)

Maximum scoring difference: 0.0
Average scoring difference: 0.0


In [31]:
# ============================================================
# TEST MULTIPLE LEAGUE SCORING SYSTEMS
# ============================================================

half_ppr_settings = DEFAULT_LEAGUE_SETTINGS.copy()

full_ppr_settings = DEFAULT_LEAGUE_SETTINGS.copy()
full_ppr_settings['reception_points'] = 1.0

standard_settings = DEFAULT_LEAGUE_SETTINGS.copy()
standard_settings['reception_points'] = 0.0


def compare_league_scoring(player_row):

    results = []

    leagues = {
        'Your Yahoo League (0.5 PPR)': half_ppr_settings,
        'Full PPR League': full_ppr_settings,
        'Standard League (0 PPR)': standard_settings
    }

    for league_name, settings in leagues.items():

        points = score_player(
            player_row,
            settings
        )

        results.append({
            'League': league_name,
            'Fantasy Points': points
        })

    return pd.DataFrame(results)


# Find a player-game with several receptions so
# the scoring difference is easy to see
example_player = (
    recent[
        recent['receptions'] >= 8
    ]
    .iloc[0]
)

print(
    example_player['player_display_name'],
    "|",
    example_player['season'],
    "Week",
    example_player['week']
)

compare_league_scoring(
    example_player
)

Rob Gronkowski | 2021 Week 1


,League,Fantasy Points
0,Your Yahoo League (0.5 PPR),25.0
1,Full PPR League,29.0
2,Standard League (0 PPR),21.0


In [32]:
# ============================================================
# RB STAT-PREDICTION PROOF OF CONCEPT
# ============================================================

rb_data = model_data[
    (model_data['position'] == 'RB') &
    (model_data['my_fantasy_points_last3'].notna())
].copy()

# Train on 2021-2023, test on 2024
rb_train = rb_data[
    rb_data['season'] <= 2023
].copy()

rb_test = rb_data[
    rb_data['season'] == 2024
].copy()

# Use the same pregame features that already worked well
rb_stat_features = final_features['RB']

# Stats we want to predict separately
rb_targets = [
    'carries',
    'rushing_yards',
    'targets',
    'receptions',
    'receiving_yards',
    'rushing_tds',
    'receiving_tds'
]

print("RB training rows:", len(rb_train))
print("RB test rows:", len(rb_test))
print("Targets:", rb_targets)

RB training rows: 3660
RB test rows: 1208
Targets: ['carries', 'rushing_yards', 'targets', 'receptions', 'receiving_yards', 'rushing_tds', 'receiving_tds']


In [33]:
# ============================================================
# TRAIN ONE MODEL PER RB STAT
# ============================================================

rb_stat_models = {}
rb_stat_medians = {}
rb_stat_results = []

X_train = rb_train[
    rb_stat_features
].copy()

X_test = rb_test[
    rb_stat_features
].copy()

medians = X_train.median(
    numeric_only=True
)

X_train = X_train.fillna(
    medians
)

X_test = X_test.fillna(
    medians
)

for target in rb_targets:

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        rb_train[target]
    )

    preds = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        rb_test[target],
        preds
    )

    rb_stat_models[target] = model

    rb_stat_results.append({
        'Stat': target,
        'MAE': round(mae, 3),
        'Actual Average': round(
            rb_test[target].mean(),
            3
        ),
        'Predicted Average': round(
            preds.mean(),
            3
        )
    })

rb_stat_medians = medians

pd.DataFrame(
    rb_stat_results
)

,Stat,MAE,Actual Average,Predicted Average
0,carries,3.858,9.007,8.916
1,rushing_yards,22.413,39.618,38.220
2,targets,1.422,2.184,2.373
3,receptions,1.205,1.738,1.839
4,receiving_yards,11.499,13.123,13.463
5,rushing_tds,0.377,0.291,0.270
6,receiving_tds,0.119,0.064,0.068


In [34]:
# ============================================================
# TURN RB STAT PREDICTIONS INTO FANTASY PROJECTIONS
# ============================================================

rb_stat_predictions = rb_test[[
    'player_display_name',
    'season',
    'week',
    'recent_team',
    'opponent_team',
    'my_fantasy_points'
]].copy()

# Generate each predicted stat
for target in rb_targets:

    rb_stat_predictions[
        f'pred_{target}'
    ] = rb_stat_models[target].predict(
        X_test
    )


def score_predicted_rb(row, settings):

    predicted_stats = {
        'completions': 0,
        'passing_yards': 0,
        'passing_tds': 0,
        'interceptions': 0,

        'carries':
            row['pred_carries'],

        'rushing_yards':
            row['pred_rushing_yards'],

        'rushing_tds':
            row['pred_rushing_tds'],

        'targets':
            row['pred_targets'],

        'receptions':
            row['pred_receptions'],

        'receiving_yards':
            row['pred_receiving_yards'],

        'receiving_tds':
            row['pred_receiving_tds'],

        'passing_2pt_conversions': 0,
        'rushing_2pt_conversions': 0,
        'receiving_2pt_conversions': 0,

        'rushing_fumbles_lost': 0,
        'receiving_fumbles_lost': 0,
        'sack_fumbles_lost': 0
    }

    return score_player(
        predicted_stats,
        settings
    )


rb_stat_predictions[
    'stat_based_projection'
] = rb_stat_predictions.apply(
    lambda row: score_predicted_rb(
        row,
        DEFAULT_LEAGUE_SETTINGS
    ),
    axis=1
)

print(
    rb_stat_predictions[[
        'player_display_name',
        'week',
        'stat_based_projection',
        'my_fantasy_points'
    ]].head(10)
)

        player_display_name  week  stat_based_projection  my_fantasy_points
1484  Cordarrelle Patterson     2                   3.17                0.3
1485  Cordarrelle Patterson     3                   3.17                6.3
1486  Cordarrelle Patterson     4                   3.75                7.2
1487  Cordarrelle Patterson    10                   3.98                1.4
1488  Cordarrelle Patterson    11                   4.12                0.5
1489  Cordarrelle Patterson    12                   3.14               -0.2
1490  Cordarrelle Patterson    13                   2.32                6.8
1491  Cordarrelle Patterson    14                   3.75               -1.4
1492  Cordarrelle Patterson    15                   2.76                1.3
1493  Cordarrelle Patterson    16                   2.82                7.7


In [35]:
# ============================================================
# COMPARE STAT-BASED VS DIRECT FANTASY MODEL
# ============================================================

# Direct model predictions for the same 2024 RB games
direct_rb_model = final_models['RB']

direct_X = rb_test[
    final_features['RB']
].copy()

direct_X = direct_X.fillna(
    final_medians['RB']
)

rb_stat_predictions[
    'direct_projection'
] = direct_rb_model.predict(
    direct_X
)

# MAE
stat_mae = mean_absolute_error(
    rb_stat_predictions[
        'my_fantasy_points'
    ],
    rb_stat_predictions[
        'stat_based_projection'
    ]
)

direct_mae = mean_absolute_error(
    rb_stat_predictions[
        'my_fantasy_points'
    ],
    rb_stat_predictions[
        'direct_projection'
    ]
)

# Ranking correlation
stat_rank = spearmanr(
    rb_stat_predictions[
        'stat_based_projection'
    ],
    rb_stat_predictions[
        'my_fantasy_points'
    ]
).statistic

direct_rank = spearmanr(
    rb_stat_predictions[
        'direct_projection'
    ],
    rb_stat_predictions[
        'my_fantasy_points'
    ]
).statistic


comparison = pd.DataFrame([
    {
        'Model': 'Stat-based projection',
        'MAE': round(stat_mae, 3),
        'Rank correlation': round(
            stat_rank,
            3
        )
    },
    {
        'Model': 'Direct fantasy-points model',
        'MAE': round(direct_mae, 3),
        'Rank correlation': round(
            direct_rank,
            3
        )
    }
])

comparison

,Model,MAE,Rank correlation
0,Stat-based projection,4.690,0.597
1,Direct fantasy-points model,4.671,0.602


In [36]:
# ============================================================
# COMPARE RB START/SIT RANKING ACCURACY
# STAT-BASED VS DIRECT MODEL
# ============================================================

import itertools

pairwise_results = []

for model_name, projection_col in [
    ('Stat-based', 'stat_based_projection'),
    ('Direct', 'direct_projection')
]:

    correct = 0
    total = 0

    for week, group in rb_stat_predictions.groupby('week'):

        rows = group.to_dict('records')

        for a, b in itertools.combinations(rows, 2):

            # Ignore exact ties in actual scoring
            if (
                a['my_fantasy_points']
                == b['my_fantasy_points']
            ):
                continue

            predicted_winner = (
                'a'
                if a[projection_col]
                > b[projection_col]
                else 'b'
            )

            actual_winner = (
                'a'
                if a['my_fantasy_points']
                > b['my_fantasy_points']
                else 'b'
            )

            if predicted_winner == actual_winner:
                correct += 1

            total += 1

    pairwise_results.append({
        'Model': model_name,
        'Pairwise Accuracy': round(
            correct / total,
            4
        ),
        'Comparisons': total
    })

pd.DataFrame(pairwise_results)

,Model,Pairwise Accuracy,Comparisons
0,Stat-based,0.7120,42371
1,Direct,0.7154,42371


In [37]:
# ============================================================
# WR + TE STAT-PREDICTION PROOF OF CONCEPT
# ============================================================

receiver_results = []
receiver_stat_models = {}

receiver_targets = [
    'targets',
    'receptions',
    'receiving_yards',
    'receiving_tds',
    'carries',
    'rushing_yards',
    'rushing_tds'
]

for position in ['WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == position) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    train = pos_data[
        pos_data['season'] <= 2023
    ].copy()

    test = pos_data[
        pos_data['season'] == 2024
    ].copy()

    features = final_features[position]

    X_train = train[features].copy()
    X_test = test[features].copy()

    medians = X_train.median(
        numeric_only=True
    )

    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)

    receiver_stat_models[position] = {}

    print()
    print("=" * 50)
    print(position)
    print("=" * 50)

    print(
        "Training rows:",
        len(train),
        "| Test rows:",
        len(test)
    )

    for target in receiver_targets:

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            train[target]
        )

        preds = model.predict(
            X_test
        )

        mae = mean_absolute_error(
            test[target],
            preds
        )

        receiver_stat_models[
            position
        ][target] = model

        receiver_results.append({
            'Position': position,
            'Stat': target,
            'MAE': round(mae, 3),
            'Actual Average': round(
                test[target].mean(),
                3
            ),
            'Predicted Average': round(
                preds.mean(),
                3
            )
        })

receiver_stat_results = pd.DataFrame(
    receiver_results
)

receiver_stat_results


WR
Training rows: 5883 | Test rows: 1905

TE
Training rows: 2855 | Test rows: 969


,Position,Stat,MAE,Actual Average,Predicted Average
0,WR,targets,2.083,4.964,4.970
1,WR,receptions,1.583,3.157,3.134
2,WR,receiving_yards,23.433,39.586,39.611
3,WR,receiving_tds,0.361,0.265,0.227
4,WR,carries,0.279,0.199,0.202
5,WR,rushing_yards,1.852,1.000,1.214
6,WR,rushing_tds,0.015,0.006,0.010
7,TE,targets,1.749,3.642,3.577
8,TE,receptions,1.427,2.654,2.501
9,TE,receiving_yards,17.102,26.904,26.257


In [38]:
# ============================================================
# TURN WR + TE STAT PREDICTIONS INTO FANTASY PROJECTIONS
# ============================================================

receiver_projection_results = []

for position in ['WR', 'TE']:

    pos_data = model_data[
        (model_data['position'] == position) &
        (model_data['my_fantasy_points_last3'].notna())
    ].copy()

    test = pos_data[
        pos_data['season'] == 2024
    ].copy()

    features = final_features[position]

    X_test = test[
        features
    ].copy()

    # Use the medians from the final model
    X_test = X_test.fillna(
        final_medians[position]
    )

    prediction_df = test[[
        'player_display_name',
        'season',
        'week',
        'recent_team',
        'opponent_team',
        'my_fantasy_points'
    ]].copy()

    # Predict each underlying stat
    for target in receiver_targets:

        prediction_df[
            f'pred_{target}'
        ] = receiver_stat_models[
            position
        ][target].predict(
            X_test
        )

    # Score the predicted stat line
    def score_receiver_projection(row):

        predicted_stats = {
            'completions': 0,
            'passing_yards': 0,
            'passing_tds': 0,
            'interceptions': 0,

            'carries':
                row['pred_carries'],

            'rushing_yards':
                row['pred_rushing_yards'],

            'rushing_tds':
                row['pred_rushing_tds'],

            'targets':
                row['pred_targets'],

            'receptions':
                row['pred_receptions'],

            'receiving_yards':
                row['pred_receiving_yards'],

            'receiving_tds':
                row['pred_receiving_tds'],

            'passing_2pt_conversions': 0,
            'rushing_2pt_conversions': 0,
            'receiving_2pt_conversions': 0,

            'rushing_fumbles_lost': 0,
            'receiving_fumbles_lost': 0,
            'sack_fumbles_lost': 0
        }

        return score_player(
            predicted_stats,
            DEFAULT_LEAGUE_SETTINGS
        )

    prediction_df[
        'stat_based_projection'
    ] = prediction_df.apply(
        score_receiver_projection,
        axis=1
    )

    # Direct fantasy-points model
    prediction_df[
        'direct_projection'
    ] = final_models[
        position
    ].predict(
        X_test
    )

    # MAE
    stat_mae = mean_absolute_error(
        prediction_df[
            'my_fantasy_points'
        ],
        prediction_df[
            'stat_based_projection'
        ]
    )

    direct_mae = mean_absolute_error(
        prediction_df[
            'my_fantasy_points'
        ],
        prediction_df[
            'direct_projection'
        ]
    )

    # Ranking
    stat_rank = spearmanr(
        prediction_df[
            'stat_based_projection'
        ],
        prediction_df[
            'my_fantasy_points'
        ]
    ).statistic

    direct_rank = spearmanr(
        prediction_df[
            'direct_projection'
        ],
        prediction_df[
            'my_fantasy_points'
        ]
    ).statistic

    receiver_projection_results.append({
        'Position': position,
        'Model': 'Stat-based',
        'MAE': round(stat_mae, 3),
        'Rank correlation': round(
            stat_rank,
            3
        )
    })

    receiver_projection_results.append({
        'Position': position,
        'Model': 'Direct',
        'MAE': round(direct_mae, 3),
        'Rank correlation': round(
            direct_rank,
            3
        )
    })

receiver_projection_comparison = pd.DataFrame(
    receiver_projection_results
)

receiver_projection_comparison

,Position,Model,MAE,Rank correlation
0,WR,Stat-based,4.428,0.544
1,WR,Direct,4.425,0.542
2,TE,Stat-based,3.336,0.511
3,TE,Direct,3.342,0.505


In [41]:
# ============================================================
# QB STAT-PREDICTION PROOF OF CONCEPT
# COMBINED SETUP + TRAINING CELL
# ============================================================

qb_data = model_data[
    (model_data['position'] == 'QB') &
    (model_data['my_fantasy_points_last3'].notna())
].copy()

qb_train = qb_data[
    qb_data['season'] <= 2023
].copy()

qb_test = qb_data[
    qb_data['season'] == 2024
].copy()

qb_stat_features = final_features['QB']

qb_targets = [
    'completions',
    'passing_yards',
    'passing_tds',
    'interceptions',
    'carries',
    'rushing_yards',
    'rushing_tds'
]

print("QB training rows:", len(qb_train))
print("QB test rows:", len(qb_test))
print("Targets:", qb_targets)

# ------------------------------------------------------------
# TRAIN ONE MODEL PER QB STAT
# ------------------------------------------------------------

qb_stat_models = {}
qb_stat_results = []

X_train_qb = qb_train[
    qb_stat_features
].copy()

X_test_qb = qb_test[
    qb_stat_features
].copy()

qb_medians = X_train_qb.median(
    numeric_only=True
)

X_train_qb = X_train_qb.fillna(
    qb_medians
)

X_test_qb = X_test_qb.fillna(
    qb_medians
)

for target in qb_targets:

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_qb,
        qb_train[target]
    )

    preds = model.predict(
        X_test_qb
    )

    mae = mean_absolute_error(
        qb_test[target],
        preds
    )

    qb_stat_models[target] = model

    qb_stat_results.append({
        'Stat': target,
        'MAE': round(mae, 3),
        'Actual Average': round(
            qb_test[target].mean(),
            3
        ),
        'Predicted Average': round(
            preds.mean(),
            3
        )
    })

qb_stat_medians = qb_medians

pd.DataFrame(
    qb_stat_results
)

QB training rows: 1731
QB test rows: 586
Targets: ['completions', 'passing_yards', 'passing_tds', 'interceptions', 'carries', 'rushing_yards', 'rushing_tds']


,Stat,MAE,Actual Average,Predicted Average
0,completions,5.161,18.314,18.240
1,passing_yards,61.193,200.824,201.444
2,passing_tds,0.856,1.292,1.225
3,interceptions,0.675,0.602,0.672
4,carries,1.842,3.623,3.744
5,rushing_yards,12.870,17.065,16.905
6,rushing_tds,0.261,0.162,0.175


In [42]:
# ============================================================
# QB STAT-BASED FANTASY PROJECTION VS DIRECT MODEL
# ============================================================

qb_projection_df = qb_test[[
    'player_display_name',
    'season',
    'week',
    'recent_team',
    'opponent_team',
    'my_fantasy_points'
]].copy()


# ------------------------------------------------------------
# PREDICT EACH QB STAT
# ------------------------------------------------------------

for target in qb_targets:

    qb_projection_df[
        f'pred_{target}'
    ] = qb_stat_models[target].predict(
        X_test_qb
    )


# ------------------------------------------------------------
# SCORE THE PREDICTED QB STAT LINE
# ------------------------------------------------------------

def score_qb_projection(row):

    predicted_stats = {
        # Passing
        'completions':
            row['pred_completions'],

        'passing_yards':
            row['pred_passing_yards'],

        'passing_tds':
            row['pred_passing_tds'],

        'interceptions':
            row['pred_interceptions'],

        # Rushing
        'carries':
            row['pred_carries'],

        'rushing_yards':
            row['pred_rushing_yards'],

        'rushing_tds':
            row['pred_rushing_tds'],

        # Receiving
        'targets': 0,
        'receptions': 0,
        'receiving_yards': 0,
        'receiving_tds': 0,

        # 2-point conversions
        'passing_2pt_conversions': 0,
        'rushing_2pt_conversions': 0,
        'receiving_2pt_conversions': 0,

        # Fumbles
        'rushing_fumbles_lost': 0,
        'receiving_fumbles_lost': 0,
        'sack_fumbles_lost': 0
    }

    return score_player(
        predicted_stats,
        DEFAULT_LEAGUE_SETTINGS
    )


qb_projection_df[
    'stat_based_projection'
] = qb_projection_df.apply(
    score_qb_projection,
    axis=1
)


# ------------------------------------------------------------
# DIRECT QB MODEL
# ------------------------------------------------------------

direct_qb_X = qb_test[
    final_features['QB']
].copy()

direct_qb_X = direct_qb_X.fillna(
    final_medians['QB']
)

qb_projection_df[
    'direct_projection'
] = final_models[
    'QB'
].predict(
    direct_qb_X
)


# ------------------------------------------------------------
# COMPARE MAE + RANKING
# ------------------------------------------------------------

qb_stat_mae = mean_absolute_error(
    qb_projection_df[
        'my_fantasy_points'
    ],
    qb_projection_df[
        'stat_based_projection'
    ]
)

qb_direct_mae = mean_absolute_error(
    qb_projection_df[
        'my_fantasy_points'
    ],
    qb_projection_df[
        'direct_projection'
    ]
)

qb_stat_rank = spearmanr(
    qb_projection_df[
        'stat_based_projection'
    ],
    qb_projection_df[
        'my_fantasy_points'
    ]
).statistic

qb_direct_rank = spearmanr(
    qb_projection_df[
        'direct_projection'
    ],
    qb_projection_df[
        'my_fantasy_points'
    ]
).statistic


qb_comparison = pd.DataFrame([
    {
        'Model': 'Stat-based',
        'MAE': round(
            qb_stat_mae,
            3
        ),
        'Rank correlation': round(
            qb_stat_rank,
            3
        )
    },
    {
        'Model': 'Direct',
        'MAE': round(
            qb_direct_mae,
            3
        ),
        'Rank correlation': round(
            qb_direct_rank,
            3
        )
    }
])

qb_comparison

,Model,MAE,Rank correlation
0,Stat-based,6.370,0.498
1,Direct,6.343,0.502
